In [ ]:
# === パラメータ入力 ===

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# === プロファイル作成用 ===
N_sweep = 300
L_total_m = 30e-3          # total distance [m] (per side stage)
T_ramp_s = 5.0             # ramp-up duration [s]
v_target_mps = 40e-6       # target constant velocity [m/s] 例：80 µm/s

b_save = True
b_save_parameters = True
filename = 'exp_60mm_80umps'
output_dir = PROJECT_ROOT / 'examples' / 'profile'

# === 直径推定プロット用 ===
d0_um = 125.0     # 初期ファイバー径 d0 [µm]
H_mm = 4.322         # ホットゾーン長 H [mm]

N_points = 800    # プロット分解能
t_max_s = 700    # 時間プロット最大時間 [s]

use_log_y = False  # y軸を対数表示にするか
use_profile = True  # 速度プロファイル(npz)を使う

# === 速度一定プロット用 ===
L_max_mm = 60.0   # 総延伸長 L [mm]
v_mm_s = 0.040    # 片側ステージ移動速度 v [mm/s]

In [ ]:
import os
import numpy as np
from mpmath import mp, mpf, nint

from methods import jerk_matrices
from const import dt, dx, bits

def bin_ctrl(t):  # build 64-bit CTRL parameter
    t = int(t)
    r = 0b1010  # stages CCW, no DIV_SELECT, normal speed, no REF_SHIFT
    l = 0b1010
    return int(r << 59 | l << 54 | t).to_bytes(64 >> 3, byteorder='big', signed=False)

mp.dps = 80
mp.pretty = False
jm = jerk_matrices(dt=dt, dx=dx, bits=bits)

def bin_(i):
    return int(i).to_bytes(jm.bits >> 3, byteorder='big', signed=True)

dt_s = mpf(str(dt))   # dt, dx are mpf in your const.py
dx_m = mpf(str(dx))

v_counts_per_tick = mpf(v_target_mps) * dt_s / dx_m   # [counts/tick]
t_ramp = nint(mpf(T_ramp_s) / dt_s)                   # [tick]

# distance moved during ramp: x = 0.5*v*T  (in counts)
x_ramp = nint(mpf('0.5') * v_counts_per_tick * t_ramp)

x_total = nint(mpf(L_total_m) / dx_m)                 # [count]
x_const = x_total - x_ramp                            # [count]

if x_const <= 0:
    raise ValueError("Ramp distance exceeds total distance. Reduce T_ramp or increase L_total.")

# mpf で計算
x_total_mpf = nint(mpf(L_total_m) / dx_m)
x_const_mpf = x_total_mpf - x_ramp

# ここで確定（整数カウント）
x_total = int(x_total_mpf)
x_const = int(x_const_mpf)

# 以降は純粋な整数演算
x_step = x_const // N_sweep
x_rem  = x_const - x_step * N_sweep


if x_step <= 0:
    raise ValueError("x_step became 0. Increase L_total or reduce N_sweep, or lower T_ramp.")

# ticks per sweep (nominal)
t_step = nint(x_step / v_counts_per_tick)

# =========================
# Build profile
# =========================
profile = []
parameters = []
times = []

def append_segment(p_poly, t_ticks):
    b = jm.to_recursive(p_poly)  # 6 coeffs; you store first 5
    profile.append(bin_ctrl(t_ticks))
    for i in b[:5]:
        profile.append(bin_(i))
    parameters.append([int(x) for x in b[:5]])
    times.append(int(t_ticks))

# (1) ramp-up segment: 0 -> v_target
p_init = jm.segmented_initial(t_ramp, x_ramp, v_counts_per_tick)
append_segment(p_init, t_ramp)

# (2) constant-velocity sweeps: N_sweep segments
for k in range(N_sweep):
    # distribute remainder by adding +1 count to first x_rem sweeps
    xk = x_step + (1 if k < int(x_rem) else 0)
    tk = nint(xk / v_counts_per_tick)

    p_mid = jm.segmented_mid(tk, xk)
    append_segment(p_mid, tk)

# =========================
# Save
# =========================
os.makedirs(output_dir, exist_ok=True)

taper_path = os.path.join(output_dir, f"{filename}.taper")
with open(taper_path, "wb") as f:
    for b in profile:
        f.write(b)

npz_path = os.path.join(output_dir, f"{filename}.npz")
np.savez(
    npz_path,
    t=np.array(times, dtype=np.int64),      # 時間は64bitで十分
    p=np.array(parameters, dtype=object)    # 係数は任意精度int
)

# Quick report (optional)
L_check_m = (int(x_ramp) + int(x_const)) * float(dx)
print(f"Total distance check: {L_check_m*1e3:.3f} mm")
print(f"Ramp distance: {float(x_ramp)*float(dx)*1e3:.3f} mm, Constant distance: {float(x_const)*float(dx)*1e3:.3f} mm")
print("Saved:", taper_path)
print("Saved:", npz_path)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from const import dt, dx, bits

#npz_path = r""
data = np.load(npz_path, allow_pickle=True)

t_steps = np.array([float(x) for x in data["t"]])     # 各sweepのtick数相当
p = data["p"]
v_param = np.array([float(x) for x in p[:, 4]])       # v * 2**bits

# 物理単位へ
v_dimless = v_param / (2**bits)
v = v_dimless * float(dx) / float(dt)                 # (N,) [m/s]

N = len(v)

# sweep境界（段プロット用）
t_edges = np.r_[0.0, np.cumsum(t_steps) * float(dt)]  # (N+1,)
t_centers = 0.5 * (t_edges[:-1] + t_edges[1:])        # (N,)

# 位置
dx_each = v * (t_steps * float(dt))                   # (N,) [m]
x = np.r_[0.0, np.cumsum(dx_each)]                    # (N+1,)

# ---- 加速度・ジャーク（sweep間差分で定義：長さ整合を取る）----
dt_between = np.diff(t_centers)                       # (N-1,)
a_between = np.diff(v) / dt_between                   # (N-1,)  ※定速ならほぼ0

# プロットしやすいように N 長に拡張（先頭は0で埋める）
a = np.r_[0.0, a_between]                             # (N,)

# jerk も同様に（a_between の差分）
j_between = np.diff(a_between) / np.diff(t_centers[1:]) if N >= 3 else np.array([])
j = np.r_[0.0, 0.0, j_between] if N >= 3 else np.r_[0.0]  # (N,) 目安

# ---- plot ----
fig, ax = plt.subplots(4, 1, sharex=True, figsize=(7, 7))

ax[0].plot(t_edges, x)
ax[0].set_ylabel("x [m]")

ax[1].step(t_edges[:-1], v, where="post")
ax[1].set_ylabel("v [m/s]")

ax[2].step(t_centers, a, where="mid")
ax[2].set_ylabel("a [m/s^2]")

ax[3].step(t_centers, j, where="mid")
ax[3].set_ylabel("j [m/s^3]")
ax[3].set_xlabel("t [s]")

for axy in ax:
    axy.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ==== 計算 ====

import os
import numpy as np
import matplotlib.pyplot as plt
from const import dt, dx, bits

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

# モデル関数定義

def diameter_vs_L(d0_um, H_mm, L_mm):
    """
    指数モデル（距離）
    d(L) = d0 * exp(-L / (2H))
    """
    L_mm = np.asarray(L_mm, dtype=float)
    return d0_um * np.exp(-L_mm / (2.0 * H_mm))


def diameter_vs_t(d0_um, H_mm, v_mm_s, t_s):
    """
    指数モデル（時間）
    片側速度 v → 総延伸速度 2v
    d(t) = d0 * exp(-v t / H)
    """
    t_s = np.asarray(t_s, dtype=float)
    L_mm = 2.0 * v_mm_s * t_s
    return diameter_vs_L(d0_um, H_mm, L_mm)

# L-d プロット

L_mm = np.linspace(0, L_max_mm, N_points)
d_um = diameter_vs_L(d0_um, H_mm, L_mm)

plt.figure()
plt.plot(L_mm, d_um)
plt.xlabel("Total extension [mm]")
plt.ylabel("Fiber diameter [µm]")
plt.title(f"d(L) | d0={d0_um} µm, H={H_mm} mm")
if use_log_y:
    plt.yscale("log")
plt.show()

d_end_from_L_um = diameter_vs_L(d0_um, H_mm, L_max_mm)
print(f"d_end (from L=L_max_mm) = {d_end_from_L_um:.6f} µm")

# t-d プロット

if use_profile:
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"npz not found: {npz_path}")

    data = np.load(npz_path, allow_pickle=True)
    t_steps = np.array([float(x) for x in data["t"]])     # 各sweepのtick数
    p = data["p"]
    v_param = np.array([float(x) for x in p[:, 4]])       # v * 2**bits

    # 物理単位へ（片側）
    v_dimless = v_param / (2**bits)
    v_mps = v_dimless * float(dx) / float(dt)

    # 片側プロファイル → 総延伸速度/距離は倍
    v_total_mps = 2.0 * v_mps

    t_edges = np.r_[0.0, np.cumsum(t_steps) * float(dt)]
    dt_each = t_steps * float(dt)
    dL_each_m = v_total_mps * dt_each
    L_total_m = np.r_[0.0, np.cumsum(dL_each_m)]

    t_s = t_edges
    if t_max_s is not None:
        mask = t_s <= t_max_s
        t_s = t_s[mask]
        L_total_m = L_total_m[mask]

    d_um_t = diameter_vs_L(d0_um, H_mm, L_total_m * 1e3)
else:
    if t_max_s is None:
        t_max_s = L_max_mm / (2.0 * v_mm_s)

    t_s = np.linspace(0, t_max_s, N_points)
    d_um_t = diameter_vs_t(d0_um, H_mm, v_mm_s, t_s)

plt.figure()
plt.plot(t_s, d_um_t)
plt.xlabel("time [s]")
plt.ylabel("fiber diameter [µm]")
if use_profile:
    plt.title(f"d(t) | d0={d0_um} µm, H={H_mm} mm, profile={os.path.basename(npz_path)}")
else:
    plt.title(f"d(t) | d0={d0_um} µm, H={H_mm} mm, v={v_mm_s} mm/s")
if use_log_y:
    plt.yscale("log")
plt.show()

d_end_from_t_um = d_um_t[-1]
print(f"d_end (from t=t_max_s) = {d_end_from_t_um:.6f} µm")